# Notebook 22 — te_destination fold-safe

**Hypothèse** : dans les fraudes op_03 (CASH_OUT), le compte DESTINATAIRE est le "mule" qui collecte l'argent volé. Si un compte dest a déjà été impliqué dans des fraudes antérieures, c'est un signal fort et DIFFÉRENT de te_origin.

**Champion (nb08)** : te_origin smoothing=30, recent2 ≈ 0.3662, LB 0.3569

**Ce qu'on teste** :
- te_dest seul (baseline de comparaison)
- te_origin + te_dest (cumul)
- recent_target_rate pour DEST aussi

**Anti-fuite** : OOF KFold (5 splits) pour le train, fit_target_map sur ref pour val/test.

**Boussole CV↔LB** : LB ≈ recent2 − 0.004

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from src import config as C
from src.validation import time_folds, evaluate_ap, summarize_cv
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv")
test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy()
y_all = train[C.TARGET].to_numpy()
folds_full = list(time_folds(train[C.PERIOD]))
print(f"Train op03: {op03.sum():,} | Folds: {len(folds_full)}")

In [ ]:
EPS = 1e-6
WINDOWS = (5, 10, 20)
SM_ORIG = 30   # smoothing te_origin (champion)
SM_DEST = 30   # smoothing te_dest (à tester, même ordre de grandeur)

def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]
    f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)

def add_freq(X, src_df, ref_df):
    X = X.copy()
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        freq = ref_df[col].value_counts(normalize=True)
        X[f"freq_{col}"] = src_df[col].map(freq).fillna(0).values
    return X

def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    X = add_freq(X, df.reset_index(drop=True), ref)
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)

def base_build_with_dest_rate(df, ref):
    """Base + recent_target_rate pour DEST aussi."""
    X = base_build(df, ref)
    rt_dest = recent_target_rate(df, ref, C.DEST_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, rt_dest], axis=1)

def make_cat():
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=42, verbose=False)

print("Setup OK")

In [ ]:
# ── VARIANTE A : champion (te_origin seul) ──────────────────────────────────────
def feats_A_train(df, ref):
    X = base_build(df, ref)
    X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM_ORIG)
    return X

def feats_A_apply(df, ref):
    X = base_build(df, ref)
    mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM_ORIG)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm)
    return X

# ── VARIANTE B : te_origin + te_dest ────────────────────────────────────────────
def feats_B_train(df, ref):
    X = base_build(df, ref)
    X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM_ORIG)
    X["te_dest"]   = oof_target_encode_train(ref, C.DEST_ACCT,   C.TARGET, smoothing=SM_DEST)
    return X

def feats_B_apply(df, ref):
    X = base_build(df, ref)
    mpo, gmo = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM_ORIG)
    mpd, gmd = fit_target_map(ref, C.DEST_ACCT,   C.TARGET, smoothing=SM_DEST)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mpo, gmo)
    X["te_dest"]   = apply_target_map(df, C.DEST_ACCT,   mpd, gmd)
    return X

# ── VARIANTE C : te_origin + te_dest + recent_rate_dest ─────────────────────────
def feats_C_train(df, ref):
    X = base_build_with_dest_rate(df, ref)
    X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM_ORIG)
    X["te_dest"]   = oof_target_encode_train(ref, C.DEST_ACCT,   C.TARGET, smoothing=SM_DEST)
    return X

def feats_C_apply(df, ref):
    X = base_build_with_dest_rate(df, ref)
    mpo, gmo = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM_ORIG)
    mpd, gmd = fit_target_map(ref, C.DEST_ACCT,   C.TARGET, smoothing=SM_DEST)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mpo, gmo)
    X["te_dest"]   = apply_target_map(df, C.DEST_ACCT,   mpd, gmd)
    return X

print("Fonctions prêtes")

In [ ]:
def run_cv(feats_train_fn, feats_apply_fn, label=""):
    oof = np.zeros(len(train))
    per_fold = []
    last_model = None
    for tr_idx, va_idx in folds_full:
        tr_op = tr_idx[op03[tr_idx]]
        va_op = va_idx[op03[va_idx]]
        ref = train.iloc[tr_op]
        Xtr = feats_train_fn(train.iloc[tr_op], ref)
        Xva = feats_apply_fn(train.iloc[va_op], ref)
        m = make_cat().fit(Xtr, y_all[tr_op])
        oof[va_op] = m.predict_proba(Xva)[:, 1]
        per_fold.append(evaluate_ap(y_all[va_op], oof[va_op]))
        last_model = (m, Xtr.columns.tolist())
    r2 = np.mean(per_fold[-2:])
    last = per_fold[-1]
    print(f"{label:40s} recent2={r2:.4f} | last={last:.4f} | LB~{r2-0.004:.4f} | folds={[round(x,4) for x in per_fold]}")
    return per_fold, oof, last_model

print("Lancement CV...")
pfA, oofA, mA = run_cv(feats_A_train, feats_A_apply, "A: champion (te_origin)")
pfB, oofB, mB = run_cv(feats_B_train, feats_B_apply, "B: +te_dest")
pfC, oofC, mC = run_cv(feats_C_train, feats_C_apply, "C: +te_dest+recent_rate_dest")

print("\n=== RÉSUMÉ ===")
for lbl, pf in [("A champion", pfA), ("B +te_dest", pfB), ("C +te_dest+rr_dest", pfC)]:
    r2 = np.mean(pf[-2:])
    delta = r2 - np.mean(pfA[-2:])
    print(f"  {lbl:25s}: recent2={r2:.4f}  Δ={delta:+.4f}  LB~{r2-0.004:.4f}")

In [ ]:
# Importance des features dans le meilleur modèle
best_label = max([("A", pfA), ("B", pfB), ("C", pfC)], key=lambda x: np.mean(x[1][-2:]))[0]
print(f"Meilleure variante: {best_label}")
m_best = {"A": mA, "B": mB, "C": mC}[best_label]
model, cols = m_best
imp = pd.Series(model.get_feature_importance(), index=cols).sort_values(ascending=False)
print("\nTop 15 importances:")
print(imp.head(15).to_string())
if "te_dest" in imp.index:
    print(f"\n→ te_dest importance: {imp['te_dest']:.2f}")
    print(f"→ te_origin importance: {imp['te_origin']:.2f}")

In [ ]:
# Soumission — seulement si B ou C bat A
best_pf = {"A": pfA, "B": pfB, "C": pfC}[best_label]
if best_label == "A":
    print("RÉSULTAT: te_dest n'apporte rien. Pas de nouvelle soumission (nb08 reste champion).")
else:
    print(f"RÉSULTAT: variante {best_label} est meilleure → génération soumission")
    best_feats_train = {"B": feats_B_train, "C": feats_C_train}[best_label]
    best_feats_apply = {"B": feats_B_apply, "C": feats_C_apply}[best_label]

    from src.utils import op03_mask as _mask
    ref_full = train.iloc[np.where(op03)[0]]
    Xf = best_feats_train(ref_full, ref_full)
    yf = y_all[op03]
    final = make_cat().fit(Xf, yf)

    te_op = op03_mask(test).to_numpy()
    test_op = test.iloc[np.where(te_op)[0]]
    Xte = best_feats_apply(test_op, ref_full)
    proba = final.predict_proba(Xte)[:, 1]
    full = np.zeros(len(test))
    full[te_op] = proba

    path = make_submission(test[C.ID], full, f"22_te_dest_var{best_label}")
    sub = pd.read_csv(path)
    assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
    assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
    print(f"Soumission: {path} | proba>0: {int((sub['target']>0).sum())}")
    print(f"CV recent2 estimé LB: {np.mean(best_pf[-2:])-0.004:.4f}")

In [ ]:
# Analyse : nombre de comptes dest vus en fraude dans le train
ref_full = train.iloc[np.where(op03)[0]]
dest_fraud = ref_full.groupby(C.DEST_ACCT)[C.TARGET].agg(["sum", "count", "mean"])
print(f"Comptes dest dans train op03: {len(dest_fraud):,}")
print(f"  dont vus au moins 1x en fraude: {(dest_fraud['sum']>0).sum():,}")
print(f"  dont taux fraude >50%: {(dest_fraud['mean']>0.5).sum():,}")
print(f"  dont taux fraude =100%: {(dest_fraud['mean']==1.0).sum():,}")
print(f"\nmédiane comptes dans test op03 pas vus en train: à estimer")
te_op = op03_mask(test).to_numpy()
test_op = test.iloc[np.where(te_op)[0]]
new_dest = set(test_op[C.DEST_ACCT].unique()) - set(ref_full[C.DEST_ACCT].unique())
print(f"  comptes dest test op03 NOUVEAUX (pas en train): {len(new_dest):,} / {test_op[C.DEST_ACCT].nunique():,} ({100*len(new_dest)/test_op[C.DEST_ACCT].nunique():.1f}%)")